# Analyse du modèle Yusuf Counter-Cycle
## Pakistan et Afghanistan

Auteur : Marc Daghar
Licence : CC BY-SA 4.0

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import YusufCounterCycle
from src.gamification import SocialCreditModule
from src.monte_carlo import run_monte_carlo
from src.utils import load_parameters, compute_confidence_interval

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Chargement des paramètres

In [ ]:
pakistan_params = load_parameters('pakistan')
afghanistan_params = load_parameters('afghanistan')

print("Paramètres Pakistan :", pakistan_params['parameters'])
print("Paramètres Afghanistan :", afghanistan_params['parameters'])

## 2. Simulation comparative

In [ ]:
model = YusufCounterCycle(
    years=100,
    need=pakistan_params['parameters']['need_minimal'],
    gamma_high=pakistan_params['yusuf_parameters']['gamma_high'],
    gamma_low=pakistan_params['yusuf_parameters']['gamma_low'],
    threshold_ratio=pakistan_params['yusuf_parameters']['threshold_ratio']
)

df_yusuf, _ = model.simulate_yusuf(shock_year=40, shock_magnitude=0.4)
df_capitalist, _ = model.simulate_capitalist(
    interest_rate=pakistan_params['parameters']['interest_rate'],
    shock_year=40,
    shock_magnitude=0.4
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df_yusuf['Stock'], label='Yusuf (contre-cycle)', linewidth=2, color='#d4af37')
ax.plot(df_capitalist['Stock'], label='Capitaliste (intérêt)', linewidth=2, color='#dc143c', linestyle='--')
ax.plot(df_yusuf['Production'], label='Production', linewidth=1.5, color='#2ecc71', alpha=0.7)

ax.axvline(x=40, color='red', linestyle=':', alpha=0.5, label='Choc')
ax.set_xlabel('Années')
ax.set_ylabel('Stock (normalisé)')
ax.set_title('Comparaison des systèmes : Pakistan')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Simulations Monte Carlo

In [ ]:
results = run_monte_carlo(
    n_simulations=100,
    years=100,
    need=pakistan_params['parameters']['need_minimal'],
    interest_rate=pakistan_params['parameters']['interest_rate']
)

print("Stock moyen Yusuf :", compute_confidence_interval(results['yusuf_final_stocks']))
print("Stock moyen Capitaliste :", compute_confidence_interval(results['cap_final_stocks']))
print(f"Solvabilité Yusuf : {results['yusuf_solvency']*100:.1f}%")
print(f"Solvabilité Capitaliste : {results['cap_solvency']*100:.1f}%")